In [202]:
from symbol import continue_stmt

import numpy as np
import pandas as pd
import datetime as dt
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [203]:

# 1. Data
df = yf.download("AAPL", start="2016-01-01", end="2026-01-01",auto_adjust=False,multi_level_index=False)
df['returns'] = df['Adj Close'].pct_change()
df.dropna(inplace=True)

returns = df['returns']
# 2. Rolling AR(1)
window = 100
pred = []

for i in range(window, len(returns)):

    train = returns.iloc[i-window:i]

    model = ARIMA(train, order=(1,0,0)).fit()

    forecast = model.forecast(steps=1).iloc[0]   # get scalar
    pred.append(forecast)


[*********************100%***********************]  1 of 1 completed


In [206]:
# 3. Align actual returns
actual = returns.iloc[window:]

# 4. Convert to numpy
pred = np.array(pred)
actual = np.array(actual)

# 5. Signal
signals = np.where(pred > 0, 1, -1)

# 6. Strategy returns
strategy_returns = signals * actual

# 7. Buy & hold
buy_hold = actual

# 8. Performance
cum_strategy = np.cumprod(1 + strategy_returns)
cum_bh = np.cumprod(1 + buy_hold)

# 9. Metrics
sharpe = np.mean(strategy_returns) / np.std(strategy_returns)
direction_acc = np.mean((pred > 0) == (actual > 0))

print("Sharpe:", sharpe)
print("Direction Accuracy:", direction_acc)

Sharpe: 0.019354310504376953
Direction Accuracy: 0.5134687111479486


In [ ]:
print(actual)
print(pred)
print(signals)
print(buy_hold)